**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/13-zhang-method-from-scratch.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/zhang-method) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6M · Intrinsics, Extrinsics & Zhang's Method from Scratch (Cyrill Stachniss)

_Lecture 6, video lab · Cyrill Stachniss._

> 🎬 **Watch first:** [Camera Intrinsics and Extrinsics - 5 Minutes with Cyrill](https://www.youtube.com/watch?v=ND2fa08vxkY) (Cyrill Stachniss).
> Step-by-step notes for this video: [Tutorial 08 · Intrinsics & extrinsics](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/08_camera_calibration_videos.html#cyrill-intrinsics).

<a href="https://www.youtube.com/watch?v=ND2fa08vxkY"><img src="https://img.youtube.com/vi/ND2fa08vxkY/mqdefault.jpg" width="320" alt="Camera Intrinsics and Extrinsics - 5 Minutes with Cyrill"></a>

> 🎬 **Watch first:** [Intrinsic Camera Calibration - 5 Minutes with Cyrill](https://www.youtube.com/watch?v=26nV4oDLiqc) (Cyrill Stachniss).
> Step-by-step notes for this video: [Tutorial 08 · Intrinsic calibration](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/08_camera_calibration_videos.html#cyrill-calibration).

<a href="https://www.youtube.com/watch?v=26nV4oDLiqc"><img src="https://img.youtube.com/vi/26nV4oDLiqc/mqdefault.jpg" width="320" alt="Intrinsic Camera Calibration - 5 Minutes with Cyrill"></a>

> 🎬 **Watch first:** [Camera Calibration using Zhang's Method](https://www.youtube.com/watch?v=-9He7Nu3u8s) (Cyrill Stachniss).
> Step-by-step notes for this video: [Tutorial 08 · Zhang's method](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/08_camera_calibration_videos.html#cyrill-zhang).

<a href="https://www.youtube.com/watch?v=-9He7Nu3u8s"><img src="https://img.youtube.com/vi/-9He7Nu3u8s/mqdefault.jpg" width="320" alt="Camera Calibration using Zhang's Method"></a>

## What the videos say, in plain words

* **Extrinsics** = *where the camera is* (3 numbers for position, 3 for direction: 6 degrees of freedom).
  **Intrinsics** = *how the camera maps rays to pixels* (focal length / camera constant, scale difference, principal point,
  shear: 5 numbers). Together, a **3 × 4 projection matrix P** with **11** degrees of freedom: x = P X.
* **DLT (direct linear transform):** if you know **≥ 6 points in 3-D**, you can solve for P directly. But accurate 3-D
  points are hard to get.
* **Zhang's trick:** use a **flat checkerboard** and give each photo its own coordinate system on the board, so every corner
  has **Z = 0**. Then each photo gives a **homography H = K [r1 r2 t]**. Because r1 and r2 are columns of a rotation
  (perpendicular, same length) each photo gives **2 equations** on B = K⁻ᵀK⁻¹. **3 photos** → solve B → **Cholesky** → **K**.
* Real lenses add **non-linear distortion**, so finally everything is refined by **least squares on the reprojection error**.

You will implement all of it with NumPy and compare with OpenCV.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)

import lab6_checks as C
C.fetch_set('gopro')
print("data ready")

In [ ]:
import glob, json, os, pickle, time
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C
np.set_printoptions(precision=4, suppress=True)
print("OpenCV", cv2.__version__)

## 1. A synthetic camera (no distortion) and its projection matrix P = K [R | t]

In [ ]:
K_true = np.array([[820.0, 0, 640.0], [0, 815.0, 360.0], [0, 0, 1]])
rvec0, tvec0 = np.array([0.15, -0.3, 0.05]), np.array([-0.2, 0.1, 3.0])
R0, _ = cv2.Rodrigues(rvec0)
P_true = K_true @ np.hstack([R0, tvec0[:, None]])
print("P has", P_true.size, "numbers but only 11 degrees of freedom (it is defined up to scale)")

rng = np.random.default_rng(0)
Xw = rng.uniform(-1, 1, (12, 3))                         # 12 known 3-D points (NOT all on one plane)
x_h = (P_true @ np.hstack([Xw, np.ones((12, 1))]).T).T
uv = x_h[:, :2] / x_h[:, 2:]
uv_cv, _ = cv2.projectPoints(Xw, rvec0, tvec0, K_true, None)
print("P X matches projectPoints:", np.allclose(uv, uv_cv.reshape(-1, 2)))

## 2. DLT: recover P from ≥ 6 known 3-D points

Each point (X, u, v) gives two rows of a matrix A (with **X̃** = [X, Y, Z, 1]):

`[X̃, 0, −u X̃]` and `[0, X̃, −v X̃]` (each a 1 × 12 row). Then A p = 0, and p is the last right-singular vector of A.

In [ ]:
def dlt(Xw, uv):
    A = []
    for (X, Y, Z), (u, v) in zip(Xw, uv):
        Xh = [X, Y, Z, 1]
        # TODO: append the two rows for this point
        pass
    A = np.array(A)
    _, _, Vt = np.linalg.svd(A)
    return Vt[-1].reshape(3, 4)

P_est = dlt(Xw, uv)

In [ ]:
C.check_dlt(P_est, P_true)

In [ ]:
K_dlt, R_dlt, c_dlt = cv2.decomposeProjectionMatrix(P_est)[:3]
print("K from DLT (normalised):\n", K_dlt / K_dlt[2, 2])
print("camera centre:", (c_dlt[:3] / c_dlt[3]).ravel().round(3), " true:", (-R0.T @ tvec0).round(3))

## 3. Zhang, step 1: one homography per photo of the flat board

For a board point (X, Y, 0): **[u, v, 1]ᵀ ≃ H [X, Y, 1]ᵀ**. Same DLT idea with 9 unknowns. (Points are normalised first
for numerical stability; that part is given.)

In [ ]:
def normalise(p):
    m, s = p.mean(0), np.sqrt(2) / np.mean(np.linalg.norm(p - p.mean(0), axis=1))
    T = np.array([[s, 0, -s * m[0]], [0, s, -s * m[1]], [0, 0, 1]])
    return (np.hstack([p, np.ones((len(p), 1))]) @ T.T)[:, :2], T

def homography(XY, uv):
    a, Ta = normalise(XY); b, Tb = normalise(uv)
    A = []
    for (X, Y), (u, v) in zip(a, b):
        A.append([X, Y, 1, 0, 0, 0, -u * X, -u * Y, -u])
        A.append([0, 0, 0, X, Y, 1, -v * X, -v * Y, -v])
    Hn = np.linalg.svd(np.array(A))[2][-1].reshape(3, 3)
    H = np.linalg.inv(Tb) @ Hn @ Ta
    return H / H[2, 2]

# 15 synthetic photos of a 9 x 6 board (25 mm squares) with 0.2 px detection noise
board = C.board_points(9, 6, 25.0)
photos = []
for r, t in C.random_poses(15, seed=4, dist_mm=(350, 650), tilt_deg=40, board_mm=(200, 125)):
    p, _ = cv2.projectPoints(board, r, t, K_true, None)
    photos.append(p.reshape(-1, 2) + rng.normal(0, 0.2, (54, 2)))

H0 = homography(board[:, :2], photos[0])
H_cv, _ = cv2.findHomography(board[:, :2], photos[0], 0)

In [ ]:
C.check_homography(H0, H_cv)

## 4. Zhang, step 2: two equations per photo → B → K

With hᵢ = column i of H and

v_ij = [h1i h1j, h1i h2j + h2i h1j, h2i h2j, h3i h1j + h1i h3j, h3i h2j + h2i h3j, h3i h3j]

each photo gives **v₁₂ᵀ b = 0** and **(v₁₁ − v₂₂)ᵀ b = 0** for b = [B11, B12, B22, B13, B23, B33]. Stack all rows into V,
solve V b = 0 with the SVD, build B, and get K from the **Cholesky** factor: B = L Lᵀ, K⁻¹ = Lᵀ.

In [ ]:
def v_ij(H, i, j):
    h = H.T            # h[i] = column i of H
    # TODO: return the 6-vector v_ij from the formula above
    return np.zeros(6)

def zhang_K(Hs):
    V = []
    for H in Hs:
        V.append(v_ij(H, 0, 1))
        V.append(v_ij(H, 0, 0) - v_ij(H, 1, 1))
    b = np.linalg.svd(np.array(V))[2][-1]
    B = np.array([[b[0], b[1], b[3]], [b[1], b[2], b[4]], [b[3], b[4], b[5]]])
    if B[0, 0] < 0: B = -B                 # B is only known up to sign (and scale)
    L = np.linalg.cholesky(B)              # B = L L^T
    K = np.linalg.inv(L.T)
    return K / K[2, 2]

Hs = [homography(board[:, :2], p) for p in photos]
try:
    K_zhang = zhang_K(Hs)
except np.linalg.LinAlgError as e:
    K_zhang = np.full((3, 3), np.nan); print("Cholesky failed (is v_ij implemented?)", e)
print("K (Zhang, closed form) =\n", K_zhang); print("K_true =\n", K_true)

In [ ]:
C.check_zhang_K(K_zhang, K_true)

**Skew:** OpenCV assumes zero skew; this closed form estimates it, so K[0, 1] is a small number, not exactly 0.

## 5. How many photos? (the plot from Zhang's paper, in the video)

In [ ]:
noise = 0.5
counts, err = [3, 4, 5, 7, 10, 15], []
for n in counts:
    e = []
    for trial in range(20):
        sel = [p + rng.normal(0, noise, p.shape) for p in photos[:n]]
        try:
            Kz = zhang_K([homography(board[:, :2], p) for p in sel]); e.append(abs(Kz[0, 0] - K_true[0, 0]) / K_true[0, 0])
        except np.linalg.LinAlgError:
            e.append(np.nan)
    err.append(100 * np.nanmedian(e))
plt.figure(figsize=(8, 4)); plt.plot(counts, err, "o-"); plt.xlabel("number of photos"); plt.ylabel("median fx error (%)")
plt.title(f"Zhang closed form, {noise} px noise"); plt.grid(alpha=0.3); plt.show()

## 6. Real lens: closed form → non-linear refinement

On the GoPro the lens bends lines, which Zhang's linear model cannot describe, so the closed-form K is only a **start**.
OpenCV then minimises the **reprojection error** over K, distortion and all poses (Levenberg–Marquardt).

In [ ]:
op, ip = [], []
for f in sorted(glob.glob("data/calibration_wide/GOPR*.jpg")):
    ok, c = C.find_corners(cv2.imread(f, 0), (8, 6))
    if ok: op.append(C.board_points(8, 6)); ip.append(c.reshape(-1, 2))
K_lin = zhang_K([homography(o[:, :2], i) for o, i in zip(op, ip)])
rms_cv, K_cv, D_cv, _, _ = cv2.calibrateCamera(op, [i.reshape(-1, 1, 2) for i in ip], (1280, 960), None, None)
print("closed form (no distortion) fx = %.1f, cx = %.1f, cy = %.1f" % (K_lin[0, 0], K_lin[0, 2], K_lin[1, 2]))
print("OpenCV refined (with distortion) fx = %.1f, cx = %.1f, cy = %.1f, RMS %.2f px, k1 = %.3f" % (K_cv[0, 0], K_cv[0, 2], K_cv[1, 2], rms_cv, D_cv[0, 0]))

## Your turn

1. In section 2, use only **5** points. What happens, and why does Cyrill say "six or more"?
2. In section 4, make all photos **frontal** (`tilt_deg=1` in `C.random_poses`). Why does the closed form fail? Relate it to
   the two equations per photo.
3. Explain in your own words, in 3 sentences, the difference between **camera calibration** and **camera localization**
   (first video).